# AethyxLM — Production Training on Google Colab (T4)

This notebook runs the current **31.2M-parameter, 32K-vocabulary** model with the production configuration. It uses Colab for compute and Google Drive for persistent checkpoints and logs.

Before running, place every active `*.bin` file referenced by `configs/datasets.json` in `MyDrive/AethyxLM/data/`. The matching tokenizer and metadata are versioned with the repository; the tokenizer is **not** retrained here.


In [ ]:
# 1. Mount Drive, clone/update the repository, and install non-PyTorch dependencies
from google.colab import drive
from pathlib import Path
import json, os, shutil, subprocess, sys, time

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/AethyxLM')
DRIVE_DATA = DRIVE_ROOT / 'data'
DRIVE_CHECKPOINTS = DRIVE_ROOT / 'checkpoints'
DRIVE_LOGS = DRIVE_ROOT / 'logs'
DRIVE_CONFIGS = DRIVE_ROOT / 'configs'
for path in (DRIVE_DATA, DRIVE_CHECKPOINTS, DRIVE_LOGS, DRIVE_CONFIGS):
    path.mkdir(parents=True, exist_ok=True)

REPO_URL = 'https://github.com/aethyx-ai/AethyxLM.git'
REPO_ROOT = Path('/content/AETHYXLabs')
PROJECT_ROOT = REPO_ROOT / 'AethyxLM'

if (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
elif REPO_ROOT.exists():
    raise RuntimeError(f'{REPO_ROOT} exists but is not a Git checkout; remove or rename it first.')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

if not (PROJECT_ROOT / 'train.py').is_file():
    raise FileNotFoundError(f'Expected training project at {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'tokenizers>=0.13.0', 'datasets>=2.14.0',
    'tensorboard>=2.14.0', 'tqdm>=4.65.0', 'pyyaml>=6.0'
], check=True)
print(f'[OK] Project ready: {PROJECT_ROOT}')
print(f'[OK] Persistent storage: {DRIVE_ROOT}')


In [ ]:
# 2. Verify the Colab GPU and CUDA-enabled PyTorch build
import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. In Colab, select Runtime > Change runtime type > T4 GPU.')

gpu = torch.cuda.get_device_properties(0)
print(f'PyTorch: {torch.__version__}')
print(f'CUDA runtime: {torch.version.cuda}')
print(f'GPU: {gpu.name}')
print(f'VRAM: {gpu.total_memory / 2**30:.1f} GiB')
if 'T4' not in gpu.name:
    print('[INFO] This notebook is tuned for a T4; the available CUDA GPU will still be used.')


In [ ]:
# 3. Copy the storage-bounded token binaries from Drive to Colab's faster local disk
registry_path = PROJECT_ROOT / 'configs' / 'datasets.json'
registry = json.loads(registry_path.read_text(encoding='utf-8'))
binary_paths = sorted({entry[split] for entry in registry.values() for split in ('train', 'val')})

missing = []
for relative in binary_paths:
    source = DRIVE_ROOT / relative
    if not source.is_file() or source.stat().st_size == 0:
        missing.append(str(source))

if missing:
    preview = '\n'.join(f'  - {path}' for path in missing)
    raise FileNotFoundError(
        'Upload the following prepared binary files to Google Drive before continuing:\n' + preview
    )

required_bytes = sum((DRIVE_ROOT / relative).stat().st_size for relative in binary_paths)
free_bytes = shutil.disk_usage('/content').free
if required_bytes > free_bytes:
    raise RuntimeError(
        f'Need {required_bytes / 2**30:.2f} GiB locally, but only {free_bytes / 2**30:.2f} GiB is free.'
    )

for relative in binary_paths:
    source = DRIVE_ROOT / relative
    target = PROJECT_ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists() or target.stat().st_size != source.stat().st_size:
        print(f'Copying {source.name} ({source.stat().st_size / 2**20:.1f} MiB)')
        shutil.copy2(source, target)

print(f'[OK] {len(binary_paths)} binaries staged ({required_bytes / 2**30:.2f} GiB)')
subprocess.run([
    sys.executable, 'scripts/check_training_readiness.py',
    '--config', 'configs/train_config_modern.json'
], cwd=PROJECT_ROOT, check=True)


In [ ]:
# 4. Create a T4 configuration while preserving the production model and schedule
base_config_path = PROJECT_ROOT / 'configs' / 'train_config_modern.json'
config = json.loads(base_config_path.read_text(encoding='utf-8'))

# Keep the effective batch at 32 sequences while using the T4's larger memory.
config['training']['batch_size'] = 16
config['training']['grad_accum_steps'] = 2
config['training']['amp_dtype'] = 'float16'
config['data']['batch_size'] = 16
config['data']['num_workers'] = 2

# Save directly to Drive so checkpoints survive a Colab runtime reset.
config['checkpoint'].update({
    'checkpoint_dir': str(DRIVE_CHECKPOINTS),
    'log_dir': str(DRIVE_LOGS),
    'tensorboard_dir': str(DRIVE_LOGS / 'tensorboard'),
    'save_interval': 1000,
    'log_interval': 50,
})

colab_config_path = PROJECT_ROOT / 'configs' / 'train_config_colab_t4.json'
colab_config_path.write_text(json.dumps(config, indent=2) + '\n', encoding='utf-8')
shutil.copy2(colab_config_path, DRIVE_CONFIGS / colab_config_path.name)

print(f'[OK] Colab config: {colab_config_path}')
print(f"Model: {config['model']['num_layers']} layers, {config['model']['embed_dim']} dimensions")
print(f"Batch: {config['data']['batch_size']} x accumulation {config['training']['grad_accum_steps']}")
print(f"Steps: {config['training']['max_steps']:,}; checkpoint interval: {config['checkpoint']['save_interval']:,}")


In [ ]:
# 5. Detect the newest persistent checkpoint
latest_checkpoint = DRIVE_CHECKPOINTS / 'checkpoint_latest.pt'
resume_args = []

if latest_checkpoint.is_file():
    resume_args = ['--resume', str(latest_checkpoint)]
    print(f'[OK] Resuming from {latest_checkpoint}')
else:
    numbered = list(DRIVE_CHECKPOINTS.glob('checkpoint_step_*.pt'))
    if numbered:
        newest = max(numbered, key=lambda path: int(path.stem.rsplit('_', 1)[-1]))
        resume_args = ['--resume', str(newest)]
        print(f'[OK] Resuming from {newest}')
    else:
        print('[OK] No checkpoint found; starting a new run')


In [ ]:
# 6. Start or resume production training
command = [
    sys.executable, 'train.py',
    '--config', str(colab_config_path),
    '--device', 'cuda',
] + resume_args

print('Running:', ' '.join(command))
print(f'Checkpoints will be written to: {DRIVE_CHECKPOINTS}')
started = time.time()
result = subprocess.run(command, cwd=PROJECT_ROOT)
elapsed_hours = (time.time() - started) / 3600
print(f'Training process exited with code {result.returncode} after {elapsed_hours:.2f} hours')
if result.returncode != 0:
    raise RuntimeError('Training stopped with an error. Re-run the checkpoint-detection and training cells to resume.')


In [ ]:
# 7. Inspect persistent outputs
checkpoints = sorted(DRIVE_CHECKPOINTS.glob('*.pt'), key=lambda path: path.stat().st_mtime)
if not checkpoints:
    print('No checkpoints found yet.')
else:
    for path in checkpoints:
        print(f'{path.name:32s} {path.stat().st_size / 2**20:8.1f} MiB')
    print(f'[OK] Latest file by modification time: {checkpoints[-1]}')
